In [3]:
import os
import numpy as np
import neurokit2 as nk
from tqdm import tqdm
import matplotlib.pyplot as plt
import pickle
import random
import pandas as pd

### Dummy Signal Generation & Train/Val/Test Split 

In [4]:
# ==========================================
# 0. Seed
# ==========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
# ==========================================
# 1. Config
# ==========================================
TOTAL_N  = 10000
TRAIN_N  = 7000   # 70%
VAL_N    = 2000   # 20%
TEST_N   = 1000   # 10%
SEQ_LEN  = 1000
SAVE_DIR = './ProcessedData'
os.makedirs(SAVE_DIR, exist_ok=True)
# norm_range is specified per config
FILENAME_MAP = {
    'SKZFC_ART_VitalDB': {'tr': 'VitalDBTrART', 'val': 'VitalDBValART', 'test': 'VitalDBTestART', 'sig_type': 'ART', 'norm_range': (0.1, 0.6)},
    'SKZFC_ART_Mimic':   {'tr': 'Mimic3TrART',  'val': 'Mimic3ValART',  'test': 'Mimic3TestART',  'sig_type': 'ART', 'norm_range': (0.15, 0.55)},
    'SKZFC_II_VitalDB':  {'tr': 'VitalDBTrII',  'val': 'VitalDBValII',  'test': 'VitalDBTestII',  'sig_type': 'ECG', 'norm_range': (0.2, 0.35)}, 
    'SKZFC_II_Mimic':    {'tr': 'Mimic3TrII',   'val': 'Mimic3ValII',   'test': 'Mimic3TestII',   'sig_type': 'ECG', 'norm_range': (0.28, 0.48)}, 
}
# ==========================================
# 2. Dummy Signal Generator
# ==========================================
def normalize_minmax(sig, feature_range=(0, 1)):
    s_min, s_max = sig.min(), sig.max()
    if s_max - s_min == 0:
        return np.full_like(sig, feature_range[0], dtype=np.float32)
    a, b = feature_range
    return (sig - s_min) / (s_max - s_min) * (b - a) + a
def generate_dummy_signals(sig_type, num_samples, seq_len, norm_range=(0, 1)):
    signals = []
    sampling_rate = 100
    for _ in tqdm(range(num_samples), desc=f"Generating {sig_type}"):
        hr = np.random.randint(60, 95)
        if sig_type == 'ECG':
            sig = nk.ecg_simulate(length=1000, sampling_rate=sampling_rate, heart_rate=hr, method="simple")
        else:
            sig = nk.ppg_simulate(duration=1000 / sampling_rate, sampling_rate=sampling_rate, heart_rate=hr)
        if len(sig) > seq_len:
            sig = sig[:seq_len]
        elif len(sig) < seq_len:
            sig = np.pad(sig, (0, seq_len - len(sig)), 'constant')
        sig = normalize_minmax(sig, feature_range=norm_range)
        signals.append(sig)
    return np.array(signals, dtype=np.float32)
# ==========================================
# 3. Generate, Split & Save
# ==========================================
assert TRAIN_N + VAL_N + TEST_N == TOTAL_N, "TRAIN_N + VAL_N + TEST_N must equal TOTAL_N."
for ConfigName, cfg in FILENAME_MAP.items():
    tr_name   = cfg['tr']
    val_name  = cfg['val']
    test_name = cfg['test']
    sig_type  = cfg['sig_type']
    norm_range = cfg['norm_range']
    print(f"\n{'='*60}")
    print(f"[{ConfigName}] Generating {TOTAL_N} {sig_type} samples  |  norm_range: {norm_range}")
    print(f"{'='*60}")
    all_dummy_signals = generate_dummy_signals(sig_type, TOTAL_N, SEQ_LEN, norm_range=norm_range)
    tr_signals   = all_dummy_signals[:TRAIN_N]
    val_signals  = all_dummy_signals[TRAIN_N:TRAIN_N + VAL_N]
    test_signals = all_dummy_signals[TRAIN_N + VAL_N:]
    tr_path   = os.path.join(SAVE_DIR, f'{tr_name}.npy')
    val_path  = os.path.join(SAVE_DIR, f'{val_name}.npy')
    test_path = os.path.join(SAVE_DIR, f'{test_name}.npy')
    np.save(tr_path,   tr_signals)
    np.save(val_path,  val_signals)
    np.save(test_path, test_signals)
    print(f"  [Saved]  value range: [{tr_signals.min():.4f}, {tr_signals.max():.4f}]")
    print(f"  Train ({TRAIN_N}) → {tr_path}   shape: {tr_signals.shape}")
    print(f"  Val   ({VAL_N})  → {val_path}   shape: {val_signals.shape}")
    print(f"  Test  ({TEST_N}) → {test_path}  shape: {test_signals.shape}")


[SKZFC_ART_VitalDB] Generating 10000 ART samples  |  norm_range: (0.1, 0.6)


Generating ART: 100%|███████████████████████████████████████████████████████████| 10000/10000 [00:10<00:00, 987.33it/s]


  [Saved]  value range: [0.1000, 0.6000]
  Train (7000) → ./ProcessedData\VitalDBTrART.npy   shape: (7000, 1000)
  Val   (2000)  → ./ProcessedData\VitalDBValART.npy   shape: (2000, 1000)
  Test  (1000) → ./ProcessedData\VitalDBTestART.npy  shape: (1000, 1000)

[SKZFC_ART_Mimic] Generating 10000 ART samples  |  norm_range: (0.15, 0.55)


Generating ART: 100%|███████████████████████████████████████████████████████████| 10000/10000 [00:10<00:00, 955.74it/s]


  [Saved]  value range: [0.1500, 0.5500]
  Train (7000) → ./ProcessedData\Mimic3TrART.npy   shape: (7000, 1000)
  Val   (2000)  → ./ProcessedData\Mimic3ValART.npy   shape: (2000, 1000)
  Test  (1000) → ./ProcessedData\Mimic3TestART.npy  shape: (1000, 1000)

[SKZFC_II_VitalDB] Generating 10000 ECG samples  |  norm_range: (0.2, 0.35)


Generating ECG: 100%|███████████████████████████████████████████████████████████| 10000/10000 [00:12<00:00, 802.20it/s]


  [Saved]  value range: [0.2000, 0.3500]
  Train (7000) → ./ProcessedData\VitalDBTrII.npy   shape: (7000, 1000)
  Val   (2000)  → ./ProcessedData\VitalDBValII.npy   shape: (2000, 1000)
  Test  (1000) → ./ProcessedData\VitalDBTestII.npy  shape: (1000, 1000)

[SKZFC_II_Mimic] Generating 10000 ECG samples  |  norm_range: (0.28, 0.48)


Generating ECG: 100%|███████████████████████████████████████████████████████████| 10000/10000 [00:12<00:00, 794.16it/s]


  [Saved]  value range: [0.2800, 0.4800]
  Train (7000) → ./ProcessedData\Mimic3TrII.npy   shape: (7000, 1000)
  Val   (2000)  → ./ProcessedData\Mimic3ValII.npy   shape: (2000, 1000)
  Test  (1000) → ./ProcessedData\Mimic3TestII.npy  shape: (1000, 1000)
